# PVBench verification notebook

This notebook checks the benchmark of *A Multi Site Benchmark of Deep Learning Models for Photovoltaic Power
Forecasting Using Real World Data from 25 Plants* in Google Colab. It verifies an archive against its SHA-256 manifest,
runs the tests, regenerates the result tables and compares them with the stored ones, draws the figures, refits
selected plants, information sets (S0, S1, S2, S3 and the others of the paper), horizons and models with the stored
settings, and writes a report.

The data are not distributed with the code. The notebook needs an archive built with `paket_olustur.py --tur tam --ham`
from a local copy of the data; with the public code archive alone it runs the tests. Use **Runtime → Change runtime type →
GPU** for the model refits, then **Runtime → Run all**.

## 1. Open the archive

Choose the source in the settings cell:

* `"drive_id"`: the ID of a ZIP shared with you on Google Drive (the part after `/d/` in its share link). Colab asks
  for access to Google Drive (and Google Cloud) with your Google account; choose the account the ZIP was shared with.
  Only that file is downloaded.
* `"drive"`: a ZIP in your own Google Drive, given as a path relative to *My Drive*.
* `"upload"`: select the ZIP when prompted.
* `"local"`: a ZIP already dropped into the Files pane (`LOCAL_ZIP`).

The settings cell also selects the model refits of section 6.

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/pvbench")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

SOURCE = "drive"                 # "drive_id", "drive", "upload" or "local"
DRIVE_FILE_ID = ""         # SOURCE = "drive_id": ID of the shared ZIP
DRIVE_ZIP = "pvbench_tam.zip"         # SOURCE = "drive": path relative to My Drive
LOCAL_ZIP = "/content/pvbench_tam.zip"   # SOURCE = "local"

# Section 6, model refits with the stored settings. Keys are listed in that section.
RUN_REFITS = True
REFITS = ["E1-S1", "E1-S2"]
REFIT_PLANTS = [8]           # plant numbers of Table 1; [] = all 25 plants (several hours on a GPU)
REFIT_MODELS = ["gru"]       # networks; the six reference methods run with them where the experiment has them
REFIT_SEEDS = 1              # seeds 0 .. REFIT_SEEDS - 1
REFIT_HORIZONS = None        # None = the horizons of the experiment
DOWNLOAD_REPORT = True        # offer the report for download at the end

os.environ["KERAS_BACKEND"] = "torch"
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "2"


In [ ]:
if SOURCE == "drive_id":
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaIoBaseDownload

    file_id = DRIVE_FILE_ID.strip()
    if not file_id or file_id.startswith("__"):
        raise ValueError("Set DRIVE_FILE_ID to the ID in the share link of the ZIP.")
    auth.authenticate_user()
    service = build("drive", "v3", cache_discovery=False)
    try:
        meta = service.files().get(fileId=file_id, fields="name,size", supportsAllDrives=True).execute()
    except HttpError as error:
        raise RuntimeError("Google Drive did not return the file. Check DRIVE_FILE_ID and sign in with the account the ZIP was shared with: delete the runtime in the Runtime menu, then Run all again.") from error
    download_dir = Path("_download")
    download_dir.mkdir(exist_ok=True)
    package = download_dir / Path(meta["name"]).name
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with package.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request, chunksize=64 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Downloaded {status.progress():.0%} of {int(meta.get('size', 0)) / 1e6:,.0f} MB", end="\r")
    print(f"\nDownloaded {package.name}")
elif SOURCE == "drive":
    from google.colab import drive

    mount = Path("_drive")
    drive.mount(str(mount.resolve()))
    relative_zip = Path(DRIVE_ZIP)
    if relative_zip.is_absolute() or ".." in relative_zip.parts:
        raise ValueError("Use a path relative to My Drive.")
    package = mount / "MyDrive" / relative_zip
elif SOURCE == "upload":
    from google.colab import files
    from tempfile import mkdtemp

    upload_dir = Path(Path(mkdtemp(prefix="upload-", dir=".")).name)
    os.chdir(upload_dir)
    try:
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Select exactly one ZIP.")
        filename = next(iter(uploaded))
        if Path(filename).name != filename:
            raise ValueError("Expected a plain ZIP filename.")
        package = upload_dir / filename
        del uploaded
    finally:
        os.chdir(ROOT)
elif SOURCE == "local":
    package = Path(LOCAL_ZIP)
else:
    raise ValueError('SOURCE must be "drive_id", "drive", "upload" or "local".')

if not package.is_file() or package.suffix.lower() != ".zip":
    raise FileNotFoundError(f"Expected a ZIP at {package}")

In [ ]:
import csv
import hashlib
import io
import re
from pathlib import PurePosixPath
from tempfile import TemporaryDirectory
from zipfile import ZipFile

def unpack_verified(package):
    manifest = "MANIFEST.csv"
    with ZipFile(package) as archive, TemporaryDirectory(dir=".") as temporary:
        members = [item.filename for item in archive.infolist() if not item.is_dir()]
        if len(members) != len(set(members)):
            raise ValueError("The ZIP contains duplicate file names.")
        for name in members:
            path = PurePosixPath(name)
            if (path.is_absolute() or ".." in path.parts or "\\" in name
                    or str(path) != name or not path.parts):
                raise ValueError(f"Invalid member path {name!r}")
        top = {PurePosixPath(name).parts[0] for name in members}
        occupied = sorted(name for name in top if Path(name).exists() or Path(name).is_symlink())
        if occupied:
            raise FileExistsError(f"Package paths already exist. Use a fresh runtime. {occupied}")
        manifest_bytes = archive.read(manifest)
        reader = csv.DictReader(io.StringIO(manifest_bytes.decode("utf-8-sig"), newline=""))
        if reader.fieldnames != ["path", "bytes", "sha256"]:
            raise ValueError("Unexpected manifest columns.")
        entries = {}
        for row in reader:
            if row["path"] in entries or not re.fullmatch(r"[0-9a-fA-F]{64}", row["sha256"]):
                raise ValueError(f"Invalid manifest entry {row['path']!r}")
            entries[row["path"]] = (int(row["bytes"]), row["sha256"].lower())
        if set(entries) != set(members) - {manifest}:
            raise ValueError("The manifest inventory differs from the ZIP inventory.")
        stage = Path(temporary)
        for name, (expected_size, expected_hash) in entries.items():
            target = stage / name
            target.parent.mkdir(parents=True, exist_ok=True)
            size, digest = 0, hashlib.sha256()
            with archive.open(name) as source, target.open("xb") as dest:
                while chunk := source.read(1024 * 1024):
                    dest.write(chunk)
                    size += len(chunk)
                    digest.update(chunk)
            if size != expected_size or digest.hexdigest() != expected_hash:
                raise ValueError(f"Manifest mismatch for {name}")
        (stage / manifest).write_bytes(manifest_bytes)
        for name in sorted(top):
            (stage / name).rename(name)
    return len(entries)

verified_count = unpack_verified(package)
KOD, OUT, STORED = Path("kod"), Path("cikti"), Path("sonuclar")
has_runs = OUT.is_dir()
has_results = STORED.is_dir()
has_hourly = Path("veri/islenmis/saatlik").is_dir()
has_predictions = has_runs and any(OUT.glob("*/pred"))
has_raw = Path("veri/ham").is_dir()
contents = ", ".join(name for name, present in [("stored runs", has_runs), ("result tables", has_results),
                                                ("processed tables", has_hourly), ("test predictions", has_predictions),
                                                ("raw downloads", has_raw)] if present) or "code only"
print(f"{package.name}: {verified_count:,} files verified against MANIFEST.csv; contents: {contents}")

## 2. Check the runtime

The pins in `kod/requirements.txt` describe the paper's environment (Windows, Python 3.13). The cell installs missing
packages, selects Keras 3 with the PyTorch backend and lists installed and pinned versions.

In [ ]:
import subprocess
import sys
import time
from importlib import metadata
from IPython.display import Markdown, display

print("Python", sys.version, flush=True)

def version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

if version("torch") is None:
    raise RuntimeError("Select a Colab runtime that includes PyTorch.")

required = ["lightgbm", "pvlib", "pyarrow", "python-docx", "numpy", "pandas",
            "scipy", "scikit-learn", "matplotlib", "requests", "markdown"]
install = [name for name in required if version(name) is None]
if version("keras") is None or int(version("keras").split(".")[0]) < 3:
    install.append("keras>=3")
if install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install], check=True)

def md_table(headers, rows):
    def cell(value):
        return str(value).replace("|", r"\|").replace("\n", " ")
    lines = [headers, ["---"] * len(headers), *rows]
    return "\n".join("| " + " | ".join(map(cell, row)) + " |" for row in lines)

pins = (KOD / "requirements.txt").read_text(encoding="utf-8-sig").splitlines()
version_rows = []
for line in pins:
    match = re.fullmatch(r"([\w.-]+)==([^\s#]+)", line.strip())
    if match:
        name, pinned = match.groups()
        version_rows.append((name, version(name) or "absent", pinned))
version_table = md_table(("Package", "Colab", "Paper environment"), version_rows)
display(Markdown(version_table))

backend = subprocess.check_output([
    sys.executable, "-c",
    "import keras, torch; assert keras.backend.backend() == 'torch'; "
    "print(keras.backend.backend(), '|', "
    "(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))"
], text=True).strip()
gpu = not backend.endswith("CPU")
print(backend)

## 3. Run every test file

Each file in `kod/testler` runs as a separate Python process; its exit code is its result. Logs are kept.

In [ ]:
from tempfile import mkdtemp

work = Path("verification")
work.mkdir(exist_ok=True)
run_dir = work / Path(mkdtemp(prefix="run-", dir=work)).name

def run_python(label, args, *, cpu=False, cwd=None, extra_env=None):
    env = os.environ.copy()
    if cpu:
        env["CUDA_VISIBLE_DEVICES"] = ""
    if extra_env:
        env.update(extra_env)
    log = (run_dir / f"{label}.log").resolve()
    start = time.monotonic()
    print(f"Running {label} ...", flush=True)
    with log.open("w", encoding="utf-8") as output:
        process = subprocess.run([sys.executable, *map(str, args)], env=env, cwd=cwd,
                                 stdout=output, stderr=subprocess.STDOUT)
    seconds = time.monotonic() - start
    if process.returncode:
        print(log.read_text(encoding="utf-8", errors="replace")[-2000:])
    print(f"Exit {process.returncode} in {seconds:.1f} s | {log.relative_to(ROOT)}")
    return process.returncode, seconds, log

test_files = sorted((KOD / "testler").rglob("*.py"))
test_rows = []
for test in test_files:
    label = test.relative_to(KOD / "testler").as_posix()
    rc, seconds, log = run_python("test-" + label.replace("/", "_"), [test])
    test_rows.append((label, "PASS" if rc == 0 else "FAIL", rc, f"{seconds:.1f}"))
if not test_files:
    test_rows.append(("No test files found", "FAIL", "", ""))
test_table = md_table(("Test file", "Result", "Exit", "Seconds"), test_rows)
display(Markdown(test_table))

## 4. Regenerate and compare the result tables

The runner executes the 22 result-table scripts in their original order and stops at the first failure. The last three
(hour boundaries, EPİAŞ record and excluded systems) need the raw downloads; the check of the excluded systems also
needs `veri/dislama_denetimi.json` and is skipped without it. The comparison covers the stored CSVs with an absolute
tolerance of `1e-6`, except three files that come from separate measurements and are not produced by the runner:
training and prediction times (`sure_olcumu.csv`, Supplementary Table S35), repeated fits with the same seed
(`ek_belirlenim_denemesi.csv`, Supplementary Table S44) and a recomputation check of one worker process
(`dogrulama_isci2.csv`).

In [ ]:
SEPARATE_MEASUREMENTS = ["sure_olcumu.csv", "ek_belirlenim_denemesi.csv", "dogrulama_isci2.csv"]
runner_rc = compare_rc = None
step_times, comparison_text, tables = "", "", None
if not has_runs:
    tables_status = "Unavailable: the archive holds no stored runs (code archive)."
else:
    tables = run_dir / Path(mkdtemp(prefix="tables-", dir=run_dir)).name
    runner_rc, _, runner_log = run_python("table-runner", [
        "calistir_hepsi.py", "--kod", KOD, "--cikti", OUT, "--sonuclar", tables, "--yalniz-tablolar"], cpu=True)
    step_lines = [line for line in runner_log.read_text(encoding="utf-8", errors="replace").splitlines()
                  if line.startswith(("OK    ", "FAIL  ", "SKIP  "))]
    step_times = "\n".join(step_lines)
    produced = len(list(tables.glob("*.csv")))
    skipped_steps = sum(line.startswith("SKIP  ") for line in step_lines)
    tables_status = f"Runner exit {runner_rc}; {produced} CSVs regenerated."
    if skipped_steps:
        tables_status += f" {skipped_steps} step(s) skipped (see the list)."
    if not has_raw:
        tables_status += " Without raw downloads the last three audit scripts cannot run."
    if has_results:
        compare_rc, _, compare_log = run_python("table-comparison", [
            "karsilastir.py", STORED, tables, "--tolerans", "1e-6", "--yoksay", *SEPARATE_MEASUREMENTS])
        comparison_text = compare_log.read_text(encoding="utf-8", errors="replace").strip()
print(tables_status)
print(step_times)
print(comparison_text)

## 5. Draw the figures and the manuscript tables

Figures are drawn from the stored result tables; the example-week figures also need the stored test predictions.

In [ ]:
figure_rows, paper_tables = [], "Unavailable: the archive holds no result tables."
if has_results:
    figure_dir = run_dir / Path(mkdtemp(prefix="figures-", dir=run_dir)).name
    for name in ["tasarim", "mimari", "siralar", "eslifark", "bilgi", "ufuk", "boyut", "onem", "gorulmemis",
                 "tumsantraller", "ornek"]:
        if name in {"tumsantraller", "ornek"} and not (OUT / "ana_icon" / "pred").is_dir():
            figure_rows.append((name, "UNAVAILABLE", "requires cikti/ana_icon/pred"))
            continue
        rc, _, log = run_python("figure-" + name, [KOD / "sekiller.py", OUT, STORED, figure_dir, name], cpu=True)
        stem = "sekil_" + ("tum_santraller" if name == "tumsantraller" else name)
        products = [figure_dir / f"{stem}.{ext}" for ext in ("pdf", "png")]
        ok = rc == 0 and all(p.is_file() and p.stat().st_size > 0 for p in products)
        figure_rows.append((name, "PASS" if ok else "FAIL", f"{stem}.pdf/.png" if ok else log.name))
    paper_dir = run_dir / "paper_tables"
    rc, _, _ = run_python("paper-tables", [KOD / "makale_tablolari.py", STORED, paper_dir], cpu=True)
    paper_tables = (f"Exit {rc}: " + ", ".join(sorted(p.name for p in paper_dir.glob("*.md")))) if paper_dir.is_dir() else f"Exit {rc}"
figure_table = md_table(("Figure", "Result", "Output or log"), figure_rows) if figure_rows else "No figures drawn."
display(Markdown(figure_table))
print("Manuscript tables:", paper_tables)

## 6. Refit models with the stored settings

Each key runs the benchmark's own training code (`python -m pvbench.experiment --stage main`) for the chosen plants,
with the settings selected in the paper's tuning stage, and compares the new test nRMSE with the stored run of the same
plant, horizon, information set, model and seed. The reference methods (persistence, seasonal persistence, smart
persistence, climatology, ridge regression, LightGBM) run with every key that has them.

| Key | Information set | Settings | Stored runs | Notes |
|---|---|---|---|---|
| E1-S1 | S1 (reanalysis up to the origin) | `cikti/ayar` | `cikti/ana` | main benchmark |
| E1-S2 | S2 (archived ICON forecast) | `cikti/ayar_icon` | `cikti/ana_icon` | main benchmark |
| E2-S0, E2-S3 | S0 (power and solar geometry), S3 (reanalysis at the target hours) | `cikti/ayar_e2` | `cikti/ana_e2` | networks: gru, lstm, tcn, cnn_gru |
| E2-S0m, E2-S1m, E2-S3m, E2-S12 | matched samples; S12 = S1 + S2 | `cikti/ayar_e2m` | `cikti/ana_e2m` | networks: gru, lstm, tcn, cnn_gru |
| E2-S2d2 | forecast issued 48 h earlier | – | `cikti/ana_s2d2` | reference methods only |
| E3-S1, E3-S2 | every network at about 50,000 parameters | `cikti/ayar`, `cikti/ayar_icon` | `cikti/ana_e3` | networks only |
| E5-S1a | S1 + 3 h moving average of power | `cikti/ayar` | `cikti/ana_e5ma` | networks: gru, cnn_lstm_gru |
| E9-S1, E9-S2 | horizons 2, 3, 6, 12 and 18 h | `cikti/ayar`, `cikti/ayar_icon` | `cikti/ana_e9` | networks: gru, lstm, tcn, cnn_gru |

Reference methods and ridge regression are deterministic; LightGBM and the networks without convolutional layers
reproduce exactly on the same hardware, while other devices and library versions give small differences. Networks with
convolutional layers vary between runs even on the same device. Without a GPU, only the reference methods are refitted.

In [ ]:
import pandas as pd

EXPERIMENTS = {
    "E1-S1": dict(info="S1", tuning="ayar", stored="ana", horizons=[1, 24]),
    "E1-S2": dict(info="S2", tuning="ayar_icon", stored="ana_icon", horizons=[1, 24]),
    "E2-S0": dict(info="S0", tuning="ayar_e2", stored="ana_e2", horizons=[1, 24]),
    "E2-S3": dict(info="S3", tuning="ayar_e2", stored="ana_e2", horizons=[1, 24]),
    "E2-S0m": dict(info="S0m", tuning="ayar_e2m", stored="ana_e2m", horizons=[1, 24]),
    "E2-S1m": dict(info="S1m", tuning="ayar_e2m", stored="ana_e2m", horizons=[1, 24]),
    "E2-S3m": dict(info="S3m", tuning="ayar_e2m", stored="ana_e2m", horizons=[1, 24]),
    "E2-S12": dict(info="S12", tuning="ayar_e2m", stored="ana_e2m", horizons=[1, 24]),
    "E2-S2d2": dict(info="S2d2", tuning=None, stored="ana_s2d2", horizons=[1, 24], networks=False),
    "E3-S1": dict(info="S1", tuning="ayar", stored="ana_e3", horizons=[1, 24],
                  extra=["--match-params", "50000", "--no-references"]),
    "E3-S2": dict(info="S2", tuning="ayar_icon", stored="ana_e3", horizons=[1, 24],
                  extra=["--match-params", "50000", "--no-references"]),
    "E5-S1a": dict(info="S1a", tuning="ayar", stored="ana_e5ma", horizons=[1, 24]),
    "E9-S1": dict(info="S1", tuning="ayar", stored="ana_e9", horizons=[2, 3, 6, 12, 18]),
    "E9-S2": dict(info="S2", tuning="ayar_icon", stored="ana_e9", horizons=[2, 3, 6, 12, 18]),
}
KEYS = ["plant", "horizon", "info", "model", "seed"]
REFERENCE_METHODS = {"persistence", "seasonal_persistence", "smart_persistence", "climatology", "ridge", "lightgbm"}

def read_runs(folder, ok_only=True):
    parts = [pd.read_csv(p) for p in sorted(Path(folder).glob("metrics_w*.csv"))]
    if not parts:
        return pd.DataFrame(columns=KEYS + ["nrmse", "status"])
    runs = pd.concat(parts, ignore_index=True)
    runs = runs[(runs["stage"] == "main") & (runs["trial"] == 0)]
    return runs[runs["status"] == "ok"] if ok_only else runs

refit_rows, refit_notes = [], []
if not RUN_REFITS:
    refit_notes.append("Skipped (RUN_REFITS is False).")
elif not (has_hourly and has_runs):
    refit_notes.append("Unavailable: the archive holds no processed tables and stored runs.")
else:
    for key in REFITS:
        spec = EXPERIMENTS[key]
        stored = read_runs(OUT / spec["stored"])
        stored = stored[stored["info"] == spec["info"]]
        available = set(stored["model"]) - REFERENCE_METHODS
        models = [] if spec.get("networks") is False else [m for m in REFIT_MODELS if m in available]
        skipped = [m for m in REFIT_MODELS if m not in models and spec.get("networks") is not False]
        if skipped:
            refit_notes.append(f"{key}: no stored runs of {', '.join(skipped)}; not refitted.")
        if models and not gpu:
            refit_notes.append(f"{key}: no GPU, networks {', '.join(models)} not refitted.")
            models = []
        if not models and "--no-references" in spec.get("extra", []):
            refit_notes.append(f"{key}: nothing to refit.")
            continue
        horizons = REFIT_HORIZONS or spec["horizons"]
        out_dir = (run_dir / f"refit_{key}").resolve()
        command = ["-W", "ignore", "-m", "pvbench.experiment", "--stage", "main", "--out", out_dir,
                   "--info", spec["info"], "--horizons", *horizons, "--seeds", REFIT_SEEDS,
                   "--worker", 0, "--p-cores", 0]
        if REFIT_PLANTS:
            command += ["--plants", *REFIT_PLANTS]
        if spec.get("tuning"):
            command += ["--tuning", (OUT / spec["tuning"]).resolve()]
        command += ["--models", *models, *spec.get("extra", [])]
        rc, seconds, log = run_python(f"refit-{key}", command, cwd=KOD.resolve())
        written = read_runs(out_dir, ok_only=False)
        written = written[written["info"] == spec["info"]]
        for row in written[written["status"] != "ok"].itertuples():      # failed fits are recorded, not raised
            refit_notes.append(f"{key}: plant {row.plant}, h {row.horizon}, {row.model}, seed {row.seed}: {row.status}")
        new = written[written["status"] == "ok"]
        merged = new.merge(stored, on=KEYS, how="left", suffixes=("", "_stored"))
        for row in merged.itertuples():
            stored_value = getattr(row, "nrmse_stored")
            delta = row.nrmse - stored_value
            refit_rows.append((key, row.plant, row.horizon, row.info, row.model, row.seed,
                               f"{stored_value:.4f}" if pd.notna(stored_value) else "no stored run",
                               f"{row.nrmse:.4f}", f"{delta:+.4f}" if pd.notna(delta) else "",
                               "" if pd.isna(stored_value) else ("same" if int(row.n) == int(row.n_stored) else "differs"),
                               getattr(row, "device", "") if isinstance(getattr(row, "device", ""), str) else "cpu"))
        n_plants = len(REFIT_PLANTS) if REFIT_PLANTS else stored["plant"].nunique()
        n_references = 0 if "--no-references" in spec.get("extra", []) else len(REFERENCE_METHODS & set(stored["model"]))
        expected = n_plants * len(horizons) * (len(models) * REFIT_SEEDS + n_references)
        refit_notes.append(f"{key}: exit {rc}, {len(merged)} of {expected} expected fits in {seconds / 60:.1f} min.")
        if len(merged) < expected:
            refit_notes.append(f"{key}: {expected - len(merged)} expected fit(s) missing; log {log.relative_to(ROOT)}.")

refit_table = md_table(("Key", "Plant", "h", "Set", "Model", "Seed", "Stored nRMSE (%)", "Refit nRMSE (%)",
                        "Refit − stored (pp)", "Test samples", "Device"), refit_rows) if refit_rows else ""
deltas = pd.DataFrame([(r[4], float(r[8])) for r in refit_rows if r[8]], columns=["model", "delta"])
if len(deltas):
    deltas["group"] = ["reference method" if m in REFERENCE_METHODS else "network" for m in deltas["model"]]
    summary = deltas.assign(absolute=deltas["delta"].abs()).groupby("group")["absolute"].agg(["count", "max", "mean"])
    refit_summary = md_table(("Group", "Fits", "Largest |difference| (pp)", "Mean |difference| (pp)"),
                             [(g, int(r["count"]), f"{r['max']:.4f}", f"{r['mean']:.4f}") for g, r in summary.iterrows()])
else:
    refit_summary = ""
print("\n".join(refit_notes))
if refit_table:
    display(Markdown(refit_summary))
    display(Markdown(refit_table))

## 7. Save the report

The report records the archive, the versions, the test results, the table comparison, the figures and the refits. It is
saved as `colab_dogrulama_raporu.md` and `.html` beside the extracted archive and offered for download.

In [ ]:
from datetime import datetime, timezone
import markdown

passed = sum(row[1] == "PASS" for row in test_rows)
sections = [
    "# PVBench verification in Google Colab",
    f"Run time (UTC) {datetime.now(timezone.utc).isoformat(timespec='seconds')}",
    f"Archive `{package.name}`: {verified_count:,} files verified against MANIFEST.csv. Contents: {contents}.",
    f"Python `{sys.version.splitlines()[0]}`. Keras backend and device: `{backend}`.",
    "## Installed and pinned versions", version_table,
    f"## Tests ({passed}/{len(test_rows)} passed)", test_table,
    "## Result tables", tables_status,
    "```text\n" + (step_times or "-") + "\n```",
    "Comparison with the stored tables (absolute tolerance 1e-6; separately measured files not compared: "
    + ", ".join(SEPARATE_MEASUREMENTS) + "):",
    "```text\n" + (comparison_text or "-") + "\n```",
    "## Figures", figure_table, f"Manuscript tables: {paper_tables}",
    "## Model refits with the stored settings",
    "Difference = refit minus stored test nRMSE, in percentage points of rated capacity.",
    "\n".join(f"* {note}" for note in refit_notes), refit_summary, refit_table,
]
report = "\n\n".join(s for s in sections if s).rstrip() + "\n"
report_md = Path("colab_dogrulama_raporu.md")
report_md.write_text(report, encoding="utf-8")
html = markdown.markdown(report, extensions=["tables", "fenced_code"])
style = ("body{font-family:Arial,sans-serif;max-width:1100px;margin:2em auto;line-height:1.4}"
         "table{border-collapse:collapse;font-size:12px}td,th{border:1px solid #bbb;padding:3px 6px}"
         "pre{background:#f4f4f4;padding:8px;overflow-x:auto}")
report_html = Path("colab_dogrulama_raporu.html")
report_html.write_text(f"<!doctype html><meta charset='utf-8'><title>PVBench verification</title>"
                       f"<style>{style}</style>{html}", encoding="utf-8")
print(report_md.resolve(), report_html.resolve(), sep="\n")
display(Markdown(report))
if DOWNLOAD_REPORT:
    try:
        from google.colab import files
        files.download(str(report_html))
        files.download(str(report_md))
    except Exception as error:  # outside Colab or blocked downloads: the files stay in /content/pvbench
        print(f"Download not started ({error}); open the Files pane to save the report.")